# Single-run Training (notebook version)

Interactive equivalent of `scripts/train.py`. The `main()` / argparse entry
point is replaced by the **config cell** below — edit `CONFIG_PATH` / `SEED`
and run the cells top to bottom.

All shared logic (data assembly, model construction, training loop) is
imported from `src/`, so this notebook stays in sync with `train.py`.

In [2]:
# --- config cell (replaces argparse main) --------------------------------
CONFIG_PATH = "configs/ablation/lstm_stat_twohead.yaml"
SEED = 42
# -------------------------------------------------------------------------

import json, sys, time
from datetime import datetime
from pathlib import Path

import numpy as np
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data.adjacency import build_adjacency, normalize_adjacency
from src.data.dataset import build_window_loaders
from src.data.splits import split_from_dates
from src.models import build_model
from src.training import Trainer, build_loss
from src.utils import RunLogger, append_results_row, load_config, set_seed

In [4]:
# load + validate config, seed every RNG
cfg = load_config(ROOT / CONFIG_PATH, root=ROOT)
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"config={CONFIG_PATH}  seed={SEED}  device={device}")
print(json.dumps({k: v for k, v in cfg.items() if k != 'project_root'}, indent=2))

config=configs/ablation/lstm_stat_twohead.yaml  seed=42  device=cuda
{
  "data": {
    "tensor_path": "dataset/output/od_15min_tensor.npz",
    "splits_path": "dataset/output/splits.json",
    "n_zones": 263,
    "loader_device": "cpu"
  },
  "forecast": {
    "T_in": 8,
    "T_out": 4,
    "stride": 1
  },
  "model": {
    "name": "lstm",
    "output_mode": "two_head",
    "hidden_dim": 64,
    "num_layers": 2,
    "dropout": 0.0
  },
  "feature_set": "stat",
  "adjacency": "flow",
  "training": {
    "max_epochs": 50,
    "patience": 5,
    "batch_size": 64,
    "lr": 0.001,
    "weight_decay": 0.0001,
    "grad_clip": 1.0
  },
  "loss": {
    "lam1": 1.0,
    "lam2": 1.0,
    "lam3": 0.5
  },
  "eval": {
    "topk": 10
  },
  "experiments_dir": "experiments",
  "seed": 42,
  "config_path": "E:\\Workspace\\NASA_ULI\\configs\\ablation\\lstm_stat_twohead.yaml"
}


In [6]:
# load dense OD tensor + chronological splits
npz = np.load(ROOT / cfg["data"]["tensor_path"])
tensor, slot_starts = npz["tensor"], npz["slot_starts"]

splits_path = ROOT / cfg["data"]["splits_path"]
if splits_path.exists():
    meta = json.loads(splits_path.read_text())
    splits = {k: tuple(v) for k, v in meta["split_idx"].items()}
else:
    print(f"[warn] {splits_path} not found; recomputing from default dates")
    splits = split_from_dates(slot_starts)
print("tensor", tensor.shape, "| splits", splits)

tensor (64320, 263, 263) | splits {'train': (0, 43968), 'val': (43968, 49824), 'test': (49824, 52800), 'ood': (58464, 64320)}


In [7]:
# feature engineering (fit on train only) + vectorized window loaders
n_zones = cfg["data"]["n_zones"]
fc = cfg["forecast"]
loader_device = cfg["data"].get("loader_device", "cpu")
loaders, fe, scaler = build_window_loaders(
    tensor, slot_starts, splits,
    feature_set=cfg["feature_set"],
    T_in=fc["T_in"], T_out=fc["T_out"],
    batch_size=cfg["training"]["batch_size"],
    n_nodes=n_zones, stride=fc["stride"],
    device=loader_device,
)
print(f"loader_device={loader_device}  n_features={fe.n_features}")
for name, ld in loaders.items():
    print(f"  {name:5s}: {ld.n_windows} windows")

loader_device=cpu  n_features=23
  train: 43957 windows
  val  : 5845 windows
  test : 2965 windows
  ood  : 5845 windows


In [5]:
# adjacency (fit on train range; the LSTM ignores it)
A_t = None
if cfg["adjacency"] in ("flow", "geo", "hybrid"):
    try:
        A = build_adjacency(cfg["adjacency"], tensor=tensor,
                            train_range=splits["train"])
        A_t = torch.from_numpy(normalize_adjacency(A))
        print("adjacency", cfg["adjacency"], A_t.shape)
    except Exception as exc:
        print(f"[warn] adjacency '{cfg['adjacency']}' unavailable: {exc}")

adjacency flow torch.Size([263, 263])


In [6]:
# model + loss
model = build_model(cfg, n_features=fe.n_features, n_nodes=n_zones)
n_params = model.count_parameters()
lw = cfg["loss"]
loss_fn = build_loss(cfg["model"]["output_mode"],
                     lam1=lw["lam1"], lam2=lw["lam2"], lam3=lw["lam3"])
print(f"model={cfg['model']['name']} "
      f"output_mode={cfg['model']['output_mode']} params={n_params:,}")

model=lstm output_mode=two_head params=124,704


In [7]:
# run directory + persisted artifacts
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_id = (f"{cfg['model']['name']}_{cfg['feature_set']}_"
          f"{cfg['model']['output_mode']}_s{SEED}_{stamp}")
run_dir = ROOT / cfg["experiments_dir"] / "runs" / run_id
logger = RunLogger(run_dir)
logger.save_yaml("config.yaml", {k: v for k, v in cfg.items()
                                 if k != "project_root"})
scaler.save(run_dir / "scaler.json")
fe.save(run_dir / "feature_engineer.npz")
print("run_dir =", run_dir)

run_dir = e:\Workspace\NASA_ULI\experiments\runs\lstm_stat_two_head_s42_20260516_163327


In [8]:
# train (AdamW + cosine LR, grad clip, early stopping on val loss)
trainer = Trainer(model, loaders, loss_fn, cfg, run_dir, device, adjacency=A_t)
t0 = time.time()
best_val = trainer.fit()
train_time = time.time() - t0
trainer.load_best()
print(f"best val loss = {best_val:.4f}  |  train time = {train_time:.1f}s")

  epoch   1 | train 16.4342 | val 11.8655 | 21.83s
  epoch   2 | train 10.0996 | val 9.1817 | 21.65s
  epoch   3 | train 8.6078 | val 8.2706 | 21.49s
  epoch   4 | train 7.9467 | val 7.7919 | 21.5s
  epoch   5 | train 7.5717 | val 7.4822 | 21.51s
  epoch   6 | train 7.3094 | val 7.2619 | 21.56s
  epoch   7 | train 7.1186 | val 7.1043 | 21.67s
  epoch   8 | train 6.9791 | val 7.0020 | 22.04s
  epoch   9 | train 6.8685 | val 6.8883 | 21.9s
  epoch  10 | train 6.7850 | val 6.7950 | 21.83s
  epoch  11 | train 6.7147 | val 6.7320 | 21.87s
  epoch  12 | train 6.6570 | val 6.6808 | 21.87s
  epoch  13 | train 6.6114 | val 6.6468 | 21.99s
  epoch  14 | train 6.5767 | val 6.6045 | 21.91s
  epoch  15 | train 6.5490 | val 6.5879 | 21.92s
  epoch  16 | train 6.5256 | val 6.5488 | 21.92s
  epoch  17 | train 6.5033 | val 6.5431 | 22.04s
  epoch  18 | train 6.4858 | val 6.5149 | 21.95s
  epoch  19 | train 6.4707 | val 6.5153 | 22.19s
  epoch  20 | train 6.4560 | val 6.4973 | 22.14s
  epoch  21 | train

In [9]:
# evaluate on val / test / OOD, persist metrics + append the results row
val_m = trainer.evaluate(loaders["val"])
test_m = trainer.evaluate(loaders["test"])
ood_m = trainer.evaluate(loaders["ood"])
metrics = {"best_val_loss": best_val, "train_time_sec": train_time,
           "n_params": n_params, "val": val_m, "test": test_m, "ood": ood_m}
logger.save_json("metrics.json", metrics)

row = {
    "run_id": run_id, "model": cfg["model"]["name"],
    "feature_set": cfg["feature_set"],
    "output_mode": cfg["model"]["output_mode"], "seed": SEED,
    "val_mae": val_m["mae"], "val_rmse": val_m["rmse"],
    "test_mae": test_m["mae"], "test_rmse": test_m["rmse"],
    "test_masked_mae": test_m["masked_mae"],
    "test_topK_hit": test_m["topk_hit"], "test_tcs": test_m["tcs"],
    "ood_mae": ood_m["mae"], "ood_rmse": ood_m["rmse"],
    "train_time_sec": round(train_time, 1), "n_params": n_params,
}
append_results_row(ROOT / cfg["experiments_dir"] / "results.csv", row)

print(f"test  MAE={test_m['mae']:.4f} RMSE={test_m['rmse']:.4f} "
      f"topK={test_m['topk_hit']:.3f} TCS={test_m['tcs']:.3f}")
print(f"ood   MAE={ood_m['mae']:.4f} RMSE={ood_m['rmse']:.4f}")
row

test  MAE=0.1196 RMSE=0.4144 topK=0.311 TCS=0.964
ood   MAE=0.1162 RMSE=0.3944


{'run_id': 'lstm_stat_two_head_s42_20260516_163327',
 'model': 'lstm',
 'feature_set': 'stat',
 'output_mode': 'two_head',
 'seed': 42,
 'val_mae': 0.11571754852587594,
 'val_rmse': 0.3895314603020156,
 'test_mae': 0.11957518016154045,
 'test_rmse': 0.4144419546236516,
 'test_masked_mae': 1.1183447509909785,
 'test_topK_hit': 0.31098799138294647,
 'test_tcs': 0.9638067560954976,
 'ood_mae': 0.11624274353274726,
 'ood_rmse': 0.39440472393271275,
 'train_time_sec': 1096.0,
 'n_params': 124704}

In [13]:
torch.save(model.state_dict(), ROOT / 'model_output/lstm/lstm_stat_twohead.pth')

In [8]:
model = build_model(cfg, n_features=fe.n_features, n_nodes=n_zones)
model.load_state_dict(torch.load(ROOT / 'model_output/lstm/lstm_stat_twohead.pth'))
model.eval()

LSTM_OD(
  (head): TwoHead(
    (outflow): Linear(in_features=64, out_features=4, bias=True)
    (dest): Linear(in_features=64, out_features=1052, bias=True)
  )
  (lstm): LSTM(23, 64, num_layers=2, batch_first=True)
)